# Baseline modele: LSTM i Transformer

Notebook definiuje pipeline do porównania baseline modeli dla rozpoznawania krótkich komend głosowych. Nie uruchamia eksperymentów automatycznie; konfiguracje, modele, dataset, trening i zapis wyników znajdują się w modułach `scripts`.

In [5]:
import pandas as pd

from scripts import (
    DataFixedParams,
    DataGridParams,
    Experiment,
    FeatureFixedParams,
    FitFixedParams,
    FitGridParams,
    LABEL_ORDER,
    ModelGridParams,
    experiment_grid_dataframe,
    prepare_experiment_datasets,
    train_experiment,
)

LABEL_ORDER

('yes',
 'no',
 'up',
 'down',
 'left',
 'right',
 'on',
 'off',
 'stop',
 'go',
 'unknown',
 'silence')

## Konfiguracja eksperymentu


In [6]:
# Wszystkie parametry eksperymentu są zebrane w tej komórce.
# Konfiguracja poniżej jest małym, ale użytecznym testem poprawności treningu.

feature_fixed = FeatureFixedParams(
    n_mels=64,
    n_fft=512,
    hop_length=160,
    normalize=True,
)

data_grid = DataGridParams(
    train_fraction=[0.2, 0.5, 1],
    validation_fraction=1,
    test_fraction=1,
    unknown_fraction=1,
    silence_samples=2000,
    sampling_strategy="natural",
    seed=42,
)

model_grid = ModelGridParams(
    model_type=["lstm", "transformer"],
    # model_type=["transformer"],
    dropout=0.2,
    lstm_hidden_size=64,
    lstm_layers=2,
    lstm_bidirectional=True,
    transformer_d_model=64,
    transformer_heads=4,
    transformer_layers=2,
    transformer_ff_dim=128,
)

fit_fixed = FitFixedParams(
    device="cuda",
    use_tqdm=True,
    progress_backend="terminal",
    verbose=True,
    early_stopping=True,
    early_stopping_patience=5,
    early_stopping_min_delta=0.001,
)
fit_grid = FitGridParams(
    epochs=10,
    batch_size=64,
    learning_rate=1e-3,
    weight_decay=1e-4,
)

data_fixed = DataFixedParams(
    reuse_cached_dataset=False,
)

baseline_experiment = Experiment(
    name="small_functional_baseline_lstm_transformer",
    data_fixed=data_fixed,
    data_grid=data_grid,
    feature_fixed=feature_fixed,
    model_grid=model_grid,
    fit_fixed=fit_fixed,
    fit_grid=fit_grid,
)

experiment_grid_dataframe(baseline_experiment)


,experiment,data.train_fraction,data.validation_fraction,data.test_fraction,data.unknown_fraction,data.silence_samples,data.sampling_strategy,data.seed,model.model_type,model.dropout,...,model.lstm_layers,model.lstm_bidirectional,model.transformer_d_model,model.transformer_heads,model.transformer_layers,model.transformer_ff_dim,fit.epochs,fit.batch_size,fit.learning_rate,fit.weight_decay
0,small_functional_baseline_lstm_transformer,0.2,1,1,1,2000,natural,42,lstm,0.2,...,2,True,64,4,2,128,10,64,0.001,0.0001
1,small_functional_baseline_lstm_transformer,0.2,1,1,1,2000,natural,42,transformer,0.2,...,2,True,64,4,2,128,10,64,0.001,0.0001
2,small_functional_baseline_lstm_transformer,0.5,1,1,1,2000,natural,42,lstm,0.2,...,2,True,64,4,2,128,10,64,0.001,0.0001
3,small_functional_baseline_lstm_transformer,0.5,1,1,1,2000,natural,42,transformer,0.2,...,2,True,64,4,2,128,10,64,0.001,0.0001
4,small_functional_baseline_lstm_transformer,1.0,1,1,1,2000,natural,42,lstm,0.2,...,2,True,64,4,2,128,10,64,0.001,0.0001
5,small_functional_baseline_lstm_transformer,1.0,1,1,1,2000,natural,42,transformer,0.2,...,2,True,64,4,2,128,10,64,0.001,0.0001


## Uruchomienie eksperymentu

Najpierw przygotuj dane i sprawdź obiekt `prepared_data`. Dopiero kolejna komórka uruchamia trening oraz końcowy test.


In [7]:
prepared_data = prepare_experiment_datasets(baseline_experiment)


Building dataset
DATA configuration 1/3


Scanning archive: 100%|██████████| 64732/64732 [00:00<00:00, 72983.83it/s]


DATA configuration 2/3


Scanning archive: 100%|██████████| 64732/64732 [00:00<00:00, 71921.54it/s]


DATA configuration 3/3


Extracting archive: 100%|██████████| 1/1 [00:45<00:00, 45.59s/it]

  -> samples | train=10537 | validation=7008 | test=7046
     class        train  validation  test
     down          369         264   253
     go            373         260   251
     left          368         247   267
     no            371         270   252
     off           368         256   262
     on            373         257   246
     right         371         256   259
     silence       316         210   211
     stop          377         246   249
     unknown      6510        4221  4268
     up            369         260   272
     yes           372         261   256
  -> samples | train=26337 | validation=7008 | test=7046
     class        train  validation  test
     down          921         264   253
     go            931         260   251
     left          920         247   267
     no            927         270   252
     off           920         256   262
     on            932         257   246
     right         926         256   259
     silence       790 

  -> samples | train=52667 | validation=7008 | test=7046
     class        train  validation  test
     down         1842         264   253
     go           1861         260   251
     left         1839         247   267
     no           1853         270   252
     off          1839         256   262
     on           1864         257   246
     right        1852         256   259
     silence      1579         210   211
     stop         1885         246   249
     unknown     32550        4221  4268
     up           1843         260   272
     yes          1860         261   256


In [8]:
results = train_experiment(baseline_experiment, prepared_data)
results

Starting experiment: small_functional_baseline_lstm_transformer

Configuration run 1/6:
DATA (variable):
  - train_fraction: 0.2
  - validation_fraction: 1
  - test_fraction: 1
  - unknown_fraction: 1
  - silence_samples: 2000
  - sampling_strategy: natural
  - seed: 42
MODEL (variable):
  - model_type: lstm
  - dropout: 0.2
  - lstm_hidden_size: 64
  - lstm_layers: 2
  - lstm_bidirectional: True
  - transformer_d_model: 64
  - transformer_heads: 4
  - transformer_layers: 2
  - transformer_ff_dim: 128
FIT (variable):
  - epochs: 10
  - batch_size: 64
  - learning_rate: 0.001
  - weight_decay: 0.0001

Training model
Using device: cuda


Training: 100%|██████████| 10/10 [02:51<00:00, 17.18s/it, loss=0.2920, lr=0.001, val_acc=0.8876, val_loss=0.3437]


Training finished in 177.79 seconds



Configuration run 2/6:
DATA (variable):
  - train_fraction: 0.2
  - validation_fraction: 1
  - test_fraction: 1
  - unknown_fraction: 1
  - silence_samples: 2000
  - sampling_strategy: natural
  - seed: 42
MODEL (variable):
  - model_type: transformer
  - dropout: 0.2
  - lstm_hidden_size: 64
  - lstm_layers: 2
  - lstm_bidirectional: True
  - transformer_d_model: 64
  - transformer_heads: 4
  - transformer_layers: 2
  - transformer_ff_dim: 128
FIT (variable):
  - epochs: 10
  - batch_size: 64
  - learning_rate: 0.001
  - weight_decay: 0.0001

Training model
Using device: cuda


Training: 100%|██████████| 10/10 [02:58<00:00, 17.86s/it, loss=0.2293, lr=0.001, val_acc=0.8811, val_loss=0.3854]


Training finished in 184.39 seconds



Configuration run 3/6:
DATA (variable):
  - train_fraction: 0.5
  - validation_fraction: 1
  - test_fraction: 1
  - unknown_fraction: 1
  - silence_samples: 2000
  - sampling_strategy: natural
  - seed: 42
MODEL (variable):
  - model_type: lstm
  - dropout: 0.2
  - lstm_hidden_size: 64
  - lstm_layers: 2
  - lstm_bidirectional: True
  - transformer_d_model: 64
  - transformer_heads: 4
  - transformer_layers: 2
  - transformer_ff_dim: 128
FIT (variable):
  - epochs: 10
  - batch_size: 64
  - learning_rate: 0.001
  - weight_decay: 0.0001

Training model
Using device: cuda


...

Training: 100%|██████████| 10/10 [10:55<00:00, 65.59s/it, loss=0.1168, lr=0.001, val_acc=0.9456, val_loss=0.1922]


Training finished in 662.03 seconds



Configuration run 6/6:
DATA (variable):
  - train_fraction: 1
  - validation_fraction: 1
  - test_fraction: 1
  - unknown_fraction: 1
  - silence_samples: 2000
  - sampling_strategy: natural
  - seed: 42
MODEL (variable):
  - model_type: transformer
  - dropout: 0.2
  - lstm_hidden_size: 64
  - lstm_layers: 2
  - lstm_bidirectional: True
  - transformer_d_model: 64
  - transformer_heads: 4
  - transformer_layers: 2
  - transformer_ff_dim: 128
FIT (variable):
  - epochs: 10
  - batch_size: 64
  - learning_rate: 0.001
  - weight_decay: 0.0001

Training model
Using device: cuda


Training: 100%|██████████| 10/10 [10:57<00:00, 65.79s/it, loss=0.1359, lr=0.001, val_acc=0.9421, val_loss=0.2098]


Training finished in 663.64 seconds



Experiment finished | total runs = 6
Cleared experiment cache.


,run,best_epoch,epochs_trained,stopped_early,train_loss,train_accuracy,validation_loss,validation_accuracy,test_loss,test_accuracy
0,lstm_train0_2_val1_test1_lr0_001_seed42,10,10,False,0.292024,0.909557,0.343686,0.887557,0.369831,0.886744
1,trfm_train0_2_val1_test1_lr0_001_seed42,9,10,False,0.248611,0.922938,0.338839,0.897546,0.372314,0.897956
2,lstm_train0_5_val1_test1_lr0_001_seed42,9,10,False,0.180734,0.942590,0.241964,0.924943,0.254791,0.925064
3,trfm_train0_5_val1_test1_lr0_001_seed42,9,10,False,0.180483,0.944679,0.256494,0.925514,0.285492,0.925773
4,lstm_train1_val1_test1_lr0_001_seed42,10,10,False,0.116789,0.963601,0.192201,0.945634,0.187349,0.946494
5,trfm_train1_val1_test1_lr0_001_seed42,10,10,False,0.135868,0.958133,0.209820,0.942066,0.220559,0.938972
